# **1. Import Library dan Config**

In [1]:
import os
import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModel
from tensorflow.keras import mixed_precision
from sklearn.model_selection import train_test_split

# Mixed Precision untuk optimasi memori GPU
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Config Model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MAX_LEN = 300
BATCH_SIZE = 32
EPOCHS = 20
LR_INIT = 2e-5

2026-05-29 05:24:59.778420: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-29 05:25:00.207950: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-29 05:25:00.208030: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-29 05:25:00.259945: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-29 05:25:00.365855: I tensorflow/core/platform/cpu_feature_guar

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 2050, compute capability 8.6
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2026-05-29 05:25:09.806344: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-29 05:25:09.815054: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-29 05:25:09.815160: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-29 05:25:09.815864: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


# **2. Load Dataset dan Split Data**

In [2]:
# Load Dataset
df = pd.read_csv("final_dataset.csv")

# Memastikan label dalam format float32 untuk kompatibilitas dengan mixed precision
df["match_label"] = df["match_label"].astype(np.float32)

print("Info Dataset:")
print(df.info())

# Split Data (train, val, test) dengan proporsi 80:10:10
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("\nPembagian Dataset:")
print("Jumlah Data Train:", len(train_df))
print("Jumlah Data Val  :", len(val_df))
print("Jumlah Data Test :", len(test_df))

Info Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   job_desc     150000 non-null  object 
 1   resume       150000 non-null  object 
 2   match_label  150000 non-null  float32
 3   bin          150000 non-null  object 
dtypes: float32(1), object(3)
memory usage: 4.0+ MB
None

Pembagian Dataset:
Jumlah Data Train: 120000
Jumlah Data Val  : 15000
Jumlah Data Test : 15000


# **3. Check Panjang Text**

In [3]:
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Analisis panjang token dari 1000 sampel acak untuk job_desc dan resume
print("Menghitung distribusi panjang token...")
sample_df = df.sample(min(1000, len(df)))
job_lengths = sample_df['job_desc'].apply(lambda x: len(tokenizer.tokenize(str(x))))
cv_lengths = sample_df['resume'].apply(lambda x: len(tokenizer.tokenize(str(x))))

print("\n--- Analisis Panjang Token Job Desc ---")
print(f"Min: {job_lengths.min()} | Max: {job_lengths.max()} | Rata-rata: {job_lengths.mean():.0f}")

print("\n--- Analisis Panjang Token CV ---")
print(f"Min: {cv_lengths.min()} | Max: {cv_lengths.max()} | Rata-rata: {cv_lengths.mean():.0f}")

Menghitung distribusi panjang token...

--- Analisis Panjang Token Job Desc ---
Min: 10 | Max: 441 | Rata-rata: 180

--- Analisis Panjang Token CV ---
Min: 6 | Max: 143 | Rata-rata: 83


# **4. Encoding Teks dan Buat tf.data**

In [4]:
# Melakukan proses tokenisasi pada seluruh dataset
def encode_texts(texts_array):
    tokens = tokenizer(
        texts_array.tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="tf"
    )
    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"]
    }

print("Melakukan proses tokenisasi pada data Train...")
train_job_tokens = encode_texts(train_df["job_desc"])
train_cv_tokens = encode_texts(train_df["resume"])

print("Melakukan proses tokenisasi pada data Validation...")
val_job_tokens = encode_texts(val_df["job_desc"])
val_cv_tokens = encode_texts(val_df["resume"])

print("Melakukan proses tokenisasi pada data Test...")
test_job_tokens = encode_texts(test_df["job_desc"])
test_cv_tokens = encode_texts(test_df["resume"])

# Konversi Label ke format Tensor
y_train = tf.convert_to_tensor(train_df["match_label"].values, dtype=tf.float32)
y_val = tf.convert_to_tensor(val_df["match_label"].values, dtype=tf.float32)
y_test = tf.convert_to_tensor(test_df["match_label"].values, dtype=tf.float32)

# Membuat tf.data.Dataset dari token dan label
def build_tf_dataset(job_tokens, cv_tokens, labels, is_training=False):
    dataset = tf.data.Dataset.from_tensor_slices((
        (job_tokens["input_ids"], job_tokens["attention_mask"], 
         cv_tokens["input_ids"], cv_tokens["attention_mask"]), 
        labels
    ))
    if is_training:
        dataset = dataset.shuffle(10000)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_dataset = build_tf_dataset(train_job_tokens, train_cv_tokens, y_train, is_training=True)
val_dataset = build_tf_dataset(val_job_tokens, val_cv_tokens, y_val)
test_dataset = build_tf_dataset(test_job_tokens, test_cv_tokens, y_test)

Melakukan proses tokenisasi pada data Train...


2026-05-29 05:25:24.677111: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-29 05:25:24.678343: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-29 05:25:24.678400: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-29 05:25:24.970574: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-29 05:25:24.971190: I external/local_xla/xla/stream_executor

Melakukan proses tokenisasi pada data Validation...
Melakukan proses tokenisasi pada data Test...


# **5. Custom Layer**

In [ ]:
# Custom Mean Pooling Layer
@tf.keras.utils.register_keras_serializable()
class MeanPooling(tf.keras.layers.Layer):
    def call(self, inputs):
        token_embeddings, attention_mask = inputs
        mask = tf.cast(tf.expand_dims(attention_mask, axis=-1), dtype=token_embeddings.dtype)
        masked = token_embeddings * mask
        sum_embeddings = tf.reduce_sum(masked, axis=1)
        sum_mask = tf.reduce_sum(mask, axis=1)
        return sum_embeddings / tf.clip_by_value(sum_mask, 1e-9, 1e9)

# Custom L2 Normalization Layer
@tf.keras.utils.register_keras_serializable()
class L2Normalize(tf.keras.layers.Layer):
    def __init__(self, axis=1, **kwargs):
        super().__init__(**kwargs)
        self.axis = axis

    def call(self, inputs):
        return tf.nn.l2_normalize(inputs, axis=self.axis)

# Scaling Layer untuk mengubah range dari [-1, 1] ke [0, 1]
@tf.keras.utils.register_keras_serializable()
class ScaleTo01(tf.keras.layers.Layer):
    def call(self, inputs):
        return (inputs + 1.0) / 2.0

# **6. Build Model**

In [ ]:
# Membangun Functional API Model Siamese
def build_siamese_model(max_len: int = MAX_LEN, model_name: str = MODEL_NAME):
    # Input layers
    input_job_ids = tf.keras.layers.Input(shape=(max_len,), dtype=tf.int32, name="job_input_ids")
    input_job_mask = tf.keras.layers.Input(shape=(max_len,), dtype=tf.int32, name="job_attention_mask")
    input_cv_ids = tf.keras.layers.Input(shape=(max_len,), dtype=tf.int32, name="cv_input_ids")
    input_cv_mask = tf.keras.layers.Input(shape=(max_len,), dtype=tf.int32, name="cv_attention_mask")

    # Load Pre-trained BERT Model dan Freeze semua layer
    base_bert_model = TFAutoModel.from_pretrained(model_name)
    base_bert_model.trainable = True 
    base_bert_model.bert.embeddings.trainable = False
    base_bert_model.bert.pooler.trainable = False
    
    for layer in base_bert_model.bert.encoder.layer:
        layer.trainable = False

    # Unfreeze 1 layer terakhir untuk fine-tuning
    for layer in base_bert_model.bert.encoder.layer[-1:]:
        layer.trainable = True

    # Inisiasi Pooling Layer
    mean_pooling_layer = MeanPooling(name="mean_pooling")

    # Ekstraksi embedding dari BERT
    job_embedding_out = base_bert_model(input_ids=input_job_ids, attention_mask=input_job_mask)[0]
    cv_embedding_out = base_bert_model(input_ids=input_cv_ids, attention_mask=input_cv_mask)[0]

    job_pooled_vec = mean_pooling_layer([job_embedding_out, input_job_mask])
    cv_pooled_vec = mean_pooling_layer([cv_embedding_out, input_cv_mask])

    # Projection Head
    dense_layer = tf.keras.layers.Dense(128, activation="relu", name="proj_dense")
    dropout_layer = tf.keras.layers.Dropout(0.3, name="proj_dropout")
    projection_layer = tf.keras.layers.Dense(64, activation=None, name="proj_out")

    job_projected = projection_layer(dropout_layer(dense_layer(job_pooled_vec)))
    cv_projected = projection_layer(dropout_layer(dense_layer(cv_pooled_vec)))

    # L2 Normalization
    l2_normalization = L2Normalize(name="l2_normalize")
    job_normalized = l2_normalization(job_projected)
    cv_normalized = l2_normalization(cv_projected)

    # Hitung Cosine Similarity dan Scaling ke [0, 1]
    cosine_similarity = tf.keras.layers.Dot(axes=1, normalize=False, name="cosine_similarity")([job_normalized, cv_normalized])
    scaled_similarity = ScaleTo01(name="scaled_similarity")(cosine_similarity)
    final_match_score = tf.keras.layers.Activation("linear", dtype="float32", name="final_score")(scaled_similarity)

    model = tf.keras.Model(
        inputs=[input_job_ids, input_job_mask, input_cv_ids, input_cv_mask], 
        outputs=final_match_score, 
        name="ITCareerMatch_Siamese"
    )
    
    encoder_model = tf.keras.Model(
        inputs=[input_job_ids, input_job_mask], 
        outputs=job_normalized, 
        name="ITCareerMatch_Encoder"
    )

    return model, encoder_model

model, encoder_model = build_siamese_model()
model.summary()

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 37b9c886-5aed-4563-aa2b-100e40c2dd1e)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json
Retrying in 1s [Retry 1/5].
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['embeddings.position_ids']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of

Model: "ITCareerMatch_Siamese"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 job_input_ids (InputLayer)  [(None, 300)]                0         []                            
                                                                                                  
 job_attention_mask (InputL  [(None, 300)]                0         []                            
 ayer)                                                                                            
                                                                                                  
 cv_input_ids (InputLayer)   [(None, 300)]                0         []                            
                                                                                                  
 cv_attention_mask (InputLa  [(None, 300)]                0         []        

# **7. Loss Function dan TensorBoard**

In [12]:
# Loss Function Mean Squared Error (MSE) 
mse_metric_fn = tf.keras.losses.MeanSquaredError()

def compute_loss(y_true, y_pred):
    y_true = tf.expand_dims(y_true, axis=-1)
    return mse_metric_fn(y_true, y_pred)

# Setup TensorBoard
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
train_log_dir = 'logs/fit/' + current_time + '/train'
val_log_dir = 'logs/fit/' + current_time + '/val'

train_summary_writer = tf.summary.create_file_writer(train_log_dir)
val_summary_writer = tf.summary.create_file_writer(val_log_dir)

print(f"Direktori Log TensorBoard: logs/fit/{current_time}")
print("Untuk melihat grafik pelatihan, jalankan terminal baru dan ketik:")
print(f"tensorboard --logdir logs/fit")

Direktori Log TensorBoard: logs/fit/20260529-052736
Untuk melihat grafik pelatihan, jalankan terminal baru dan ketik:
tensorboard --logdir logs/fit


# **8. Optimizer dan Metrics**

In [13]:
# Menghitung total langkah pelatihan untuk scheduler
steps_per_epoch = len(train_df) // BATCH_SIZE
total_training_steps = steps_per_epoch * EPOCHS

# Cosine Decay Learning Rate Scheduler
learning_rate_scheduler = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LR_INIT,
    decay_steps=total_training_steps,
    alpha=0.1
)

# Konfigurasi Optimizer dengan Mixed Precision
optimizer = tf.keras.optimizers.AdamW(learning_rate=learning_rate_scheduler, weight_decay=1e-4)
optimizer = mixed_precision.LossScaleOptimizer(optimizer)

# Metrik untuk melacak MAE dan Akurasi
train_mae_tracker = tf.keras.metrics.MeanAbsoluteError(name="train_mae")
val_mae_tracker = tf.keras.metrics.MeanAbsoluteError(name="val_mae")

train_acc_tracker = tf.keras.metrics.BinaryAccuracy(threshold=0.5, name="train_acc")
val_acc_tracker = tf.keras.metrics.BinaryAccuracy(threshold=0.5, name="val_acc")

# **9.  Custom Training Step dengan tf.GradientTape**

In [14]:
# Training step
@tf.function
def execute_train_step(inputs, labels):
    with tf.GradientTape() as tape:
        predictions = model(inputs, training=True)
        loss_value = compute_loss(labels, predictions)
        scaled_loss_value = optimizer.get_scaled_loss(loss_value)

    # Hitung gradien
    scaled_gradients = tape.gradient(scaled_loss_value, model.trainable_variables)
    gradients = optimizer.get_unscaled_gradients(scaled_gradients)

    # Hapus gradien yang bernilai None
    grads_and_vars = [(grad, var) for grad, var in zip(gradients, model.trainable_variables) if grad is not None]
    optimizer.apply_gradients(grads_and_vars)

    # Update metrik
    train_mae_tracker.update_state(labels, predictions)
    labels_binary = tf.cast(labels >= 0.5, tf.float32)
    train_acc_tracker.update_state(labels_binary, predictions)

    return loss_value

# Validation step
@tf.function
def execute_val_step(inputs, labels):
    predictions = model(inputs, training=False)
    loss_value = compute_loss(labels, predictions)

    val_mae_tracker.update_state(labels, predictions)
    labels_binary = tf.cast(labels >= 0.5, tf.float32)
    val_acc_tracker.update_state(labels_binary, predictions)

    return loss_value

# **10. Training Loop**

In [15]:
best_val_mae = np.inf 
patience = 3
wait = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    # 1. Training
    train_losses = []
    for step, (x_batch, y_batch) in enumerate(train_dataset):
        loss = execute_train_step(x_batch, y_batch)
        train_losses.append(float(loss))
        
        if step % 200 == 0:
            print(f"Step {step} | Train Loss (MSE) {loss:.4f}")

    # 2. Validation
    val_losses = []
    for x_batch_val, y_batch_val in val_dataset:
        v_loss = execute_val_step(x_batch_val, y_batch_val)
        val_losses.append(float(v_loss))

    avg_train_loss = np.mean(train_losses)
    avg_val_loss = np.mean(val_losses)
    current_val_mae = val_mae_tracker.result()

    # 3. Print Metrics
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"Train MAE: {train_mae_tracker.result():.4f} | Val MAE: {current_val_mae:.4f}")
    print(f"Train Acc: {train_acc_tracker.result():.4f} | Val Acc: {val_acc_tracker.result():.4f}")

    # 4. TensorBoard Logging
    with train_summary_writer.as_default():
        tf.summary.scalar('loss', avg_train_loss, step=epoch)
        tf.summary.scalar('mae', train_mae_tracker.result(), step=epoch)
        tf.summary.scalar('accuracy', train_acc_tracker.result(), step=epoch)
        
    with val_summary_writer.as_default():
        tf.summary.scalar('loss', avg_val_loss, step=epoch)
        tf.summary.scalar('mae', current_val_mae, step=epoch)
        tf.summary.scalar('accuracy', val_acc_tracker.result(), step=epoch)

    # 5. Save Best Model
    if current_val_mae < best_val_mae:
        best_val_mae = current_val_mae
        wait = 0
        print(f"\nVal MAE membaik. Saving best model...")

        model.save("save_model/itcareermatch_best.keras")
        model.save_weights("save_model/itcareermatch_weights.h5")
        encoder_model.save("save_model/itcareermatch_encoder.keras")
        tokenizer.save_pretrained("save_model/itcareermatch_tokenizer")
    else:
        wait += 1
        print(f"EarlyStopping wait {wait}/{patience}")

    # 6. Early Stopping
    if wait >= patience:
        print("Early stopping triggered!")
        break

    # 7. Reset Metrics
    train_mae_tracker.reset_state()
    val_mae_tracker.reset_state()
    train_acc_tracker.reset_state()
    val_acc_tracker.reset_state()


Epoch 1/20


2026-05-29 05:27:57.112521: I external/local_xla/xla/service/service.cc:168] XLA service 0x7e103e395e40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-05-29 05:27:57.112578: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 2050, Compute Capability 8.6
2026-05-29 05:27:57.149741: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-05-29 05:27:57.231147: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
I0000 00:00:1780007277.356747    1914 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Step 0 | Train Loss (MSE) 0.1269
Step 200 | Train Loss (MSE) 0.0136
Step 400 | Train Loss (MSE) 0.0124
Step 600 | Train Loss (MSE) 0.0100
Step 800 | Train Loss (MSE) 0.0065
Step 1000 | Train Loss (MSE) 0.0136
Step 1200 | Train Loss (MSE) 0.0075
Step 1400 | Train Loss (MSE) 0.0072
Step 1600 | Train Loss (MSE) 0.0088
Step 1800 | Train Loss (MSE) 0.0077
Step 2000 | Train Loss (MSE) 0.0097
Step 2200 | Train Loss (MSE) 0.0092
Step 2400 | Train Loss (MSE) 0.0092
Step 2600 | Train Loss (MSE) 0.0054
Step 2800 | Train Loss (MSE) 0.0056
Step 3000 | Train Loss (MSE) 0.0067
Step 3200 | Train Loss (MSE) 0.0041
Step 3400 | Train Loss (MSE) 0.0054
Step 3600 | Train Loss (MSE) 0.0092
Train Loss: 0.0100 | Val Loss: 0.0051
Train MAE: 0.0741 | Val MAE: 0.0549
Train Acc: 0.9057 | Val Acc: 0.9431

Val MAE membaik. Saving best model...


/home/ulil/miniconda3/envs/env_new/lib/python3.10/site-packages/transformers/generation/tf_utils.py:465: UserWarning: `seed_generator` is deprecated and will be removed in a future version.
  warnings.warn("`seed_generator` is deprecated and will be removed in a future version.", UserWarning)



Epoch 2/20
Step 0 | Train Loss (MSE) 0.0053
Step 200 | Train Loss (MSE) 0.0035
Step 400 | Train Loss (MSE) 0.0108
Step 600 | Train Loss (MSE) 0.0037
Step 800 | Train Loss (MSE) 0.0032
Step 1000 | Train Loss (MSE) 0.0042
Step 1200 | Train Loss (MSE) 0.0047
Step 1400 | Train Loss (MSE) 0.0047
Step 1600 | Train Loss (MSE) 0.0043
Step 1800 | Train Loss (MSE) 0.0042
Step 2000 | Train Loss (MSE) 0.0051
Step 2200 | Train Loss (MSE) 0.0033
Step 2400 | Train Loss (MSE) 0.0033
Step 2600 | Train Loss (MSE) 0.0039
Step 2800 | Train Loss (MSE) 0.0061
Step 3000 | Train Loss (MSE) 0.0075
Step 3200 | Train Loss (MSE) 0.0036
Step 3400 | Train Loss (MSE) 0.0036
Step 3600 | Train Loss (MSE) 0.0050
Train Loss: 0.0047 | Val Loss: 0.0033
Train MAE: 0.0519 | Val MAE: 0.0440
Train Acc: 0.9336 | Val Acc: 0.9539

Val MAE membaik. Saving best model...

Epoch 3/20
Step 0 | Train Loss (MSE) 0.0049
Step 200 | Train Loss (MSE) 0.0038
Step 400 | Train Loss (MSE) 0.0035
Step 600 | Train Loss (MSE) 0.0035
Step 800 | T

# **11. Evaluasi dengan Data Test**

In [16]:
# Ambil model terbaik
model.load_weights("save_model/itcareermatch_weights.h5")

# metrik untuk evaluasi pada Test Set
test_mae_tracker = tf.keras.metrics.MeanAbsoluteError()
test_acc_tracker = tf.keras.metrics.BinaryAccuracy(threshold=0.5)
test_losses_history = []

# Evaluasi pada Test Set
for x_batch_test, y_batch_test in test_dataset:
    # Prediksi Skor
    predictions_test = model(x_batch_test, training=False)
    
    # Hitung Loss
    batch_test_loss = compute_loss(y_batch_test, predictions_test)
    test_losses_history.append(float(batch_test_loss))
    
    # Pembaruan Metrik Test
    test_mae_tracker.update_state(y_batch_test, predictions_test)
    labels_test_binary = tf.cast(y_batch_test >= 0.5, tf.float32)
    test_acc_tracker.update_state(labels_test_binary, predictions_test)

print("\nEvaluasi Model")
print(f"Test Loss (MSE): {np.mean(test_losses_history):.4f}")
print(f"Test MAE       : {test_mae_tracker.result():.4f}")
print(f"Test Accuracy  : {test_acc_tracker.result() * 100:.2f}%")


Evaluasi Model
Test Loss (MSE): 0.0018
Test MAE       : 0.0335
Test Accuracy  : 96.09%
